# Frozen model test review — corrected Arsenal-excluded experiment

Inference only, using the new checkpoints. Every model receives the same 75 Arsenal test fixtures. The selected model is chosen by Arsenal validation log loss. This replaces the old review of the inclusive models.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'experiments/isolated_arsenal.py').exists())
sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'work/matplotlib'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, joblib
from IPython.display import display, Markdown, Image
from experiments import train
from experiments.isolated_arsenal import OUTPUT
RUN=OUTPUT
REVIEW=ROOT/'results/arsenal_isolated_review'
(REVIEW/'figures').mkdir(parents=True,exist_ok=True)
record=json.loads((RUN/'manifest.json').read_text())
for rel,digest in record['artifact_sha256'].items():
    assert train.digest(RUN/rel)==digest,rel
features=record['features']
plt.rcParams.update({'figure.figsize':(11,5),'axes.spines.top':False,'axes.spines.right':False})
def finish(name):
    plt.tight_layout(); plt.savefig(REVIEW/'figures'/f'{name}.png',dpi=110,bbox_inches='tight'); plt.show(); plt.close()
def load(population,split):
    folder=RUN/population/split
    return (pd.read_csv(folder/'X.csv').astype(np.float32),pd.read_csv(folder/'y.csv').target_result.map({'A':0,'D':1,'H':2}).to_numpy(),pd.read_csv(folder/'index.csv'))
def predict(name,x):
    if name.endswith('mlp'):
        return train.predict_checkpoint(RUN/'models'/f'{name}_best.pt',x).to_numpy()
    return joblib.load(RUN/'models'/f'{name}.joblib').predict_proba(x)
metrics=pd.read_csv(RUN/'evaluation/metrics.csv')
names=metrics.model.drop_duplicates().tolist()
# Source duplicate is a deterministic replication, excluded from model selection.
candidates=[n for n in names if n!='league_frozen_source_mlp']
validation=metrics.query("split=='validation'").set_index('model').loc[candidates]
selected=validation.log_loss.idxmin()


In [2]:
x,y,meta=load('arsenal','test')
actual=pd.read_csv(train.DATA/'data/matches_clean.csv')[['match_id','home_goals','away_goals']]
base=meta.merge(actual,on='match_id',how='left',validate='one_to_one');assert base.home_goals.notna().all()
folder=REVIEW/'test_review';folder.mkdir(exist_ok=True)
base.to_csv(folder/'arsenal_test_fixtures.csv',index=False)
print('Validation-selected model:',selected);display(base)
tables={};scores=[]
for name in names:
    p=predict(name,x);pred=p.argmax(1);home=base.home_team.eq('Arsenal').to_numpy()
    t=base.copy();t['actual_result']=np.array(['A','D','H'])[y];t['predicted_result']=np.array(['A','D','H'])[pred];t['correct']=pred==y
    t['actual_arsenal_result']=np.where(y==1,'Draw',np.where(np.where(home,y==2,y==0),'Win','Loss'))
    t['predicted_arsenal_result']=np.where(pred==1,'Draw',np.where(np.where(home,pred==2,pred==0),'Win','Loss'))
    t['p_win']=np.where(home,p[:,2],p[:,0]);t['p_draw']=p[:,1];t['p_loss']=np.where(home,p[:,0],p[:,2]);t['confidence']=p.max(1)
    t.to_csv(folder/f'{name}_arsenal_test_review.csv',index=False);tables[name]=t
    scores.append({'model':name,**train.score(y,p)})
    expected=metrics.query("model==@name and split=='test'").iloc[0]
    for metric,value in train.score(y,p).items(): np.testing.assert_allclose(value,expected[metric],atol=1e-12,rtol=0)
scores=pd.DataFrame(scores);scores.to_csv(folder/'metrics.csv',index=False);display(scores)

Validation-selected model: full_finetune_mlp


,match_id,date,season_start,home_team,away_team,split,home_goals,away_goals
0,519c21973743d19a,2024-08-17,2024,Arsenal,Wolves,test,2,0
1,464d111b44344fcd,2024-08-24,2024,Aston Villa,Arsenal,test,0,2
2,e2a41210638f2075,2024-08-31,2024,Arsenal,Brighton,test,1,1
3,6a5309936cc57b7b,2024-09-15,2024,Tottenham,Arsenal,test,0,1
4,fd02e32795fd8d5c,2024-09-22,2024,Manchester City,Arsenal,test,2,2
...,...,...,...,...,...,...,...,...
70,8e64fca3f72df0f5,2026-04-25,2025,Arsenal,Newcastle,test,1,0
71,4db27d58ce8149a9,2026-05-02,2025,Arsenal,Fulham,test,3,0
72,99850dc1ece2e6df,2026-05-10,2025,West Ham,Arsenal,test,0,1
73,16f20bfde417325b,2026-05-18,2025,Arsenal,Burnley,test,1,0


/opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


,model,log_loss,brier_multiclass,accuracy,n
0,arsenal_mlp,0.959723,0.569044,0.600000,75
1,league_mlp,0.908240,0.533728,0.613333,75
2,full_finetune_mlp,0.908132,0.533687,0.600000,75
3,league_frozen_source_mlp,0.908240,0.533728,0.613333,75
4,head_finetune_mlp,0.908314,0.533759,0.613333,75
5,league_random_forest,0.949473,0.556547,0.613333,75
6,league_dummy,1.093786,0.663753,0.400000,75


In [3]:
for name,table in tables.items():
    display(Markdown('### '+name))
    with pd.option_context('display.max_rows',None): display(table)

### arsenal_mlp

,match_id,date,season_start,home_team,away_team,split,home_goals,away_goals,actual_result,predicted_result,correct,actual_arsenal_result,predicted_arsenal_result,p_win,p_draw,p_loss,confidence
0,519c21973743d19a,2024-08-17,2024,Arsenal,Wolves,test,2,0,H,H,True,Win,Win,0.886773,0.079903,0.033324,0.886773
1,464d111b44344fcd,2024-08-24,2024,Aston Villa,Arsenal,test,0,2,A,A,True,Win,Win,0.490891,0.263100,0.246009,0.490891
2,e2a41210638f2075,2024-08-31,2024,Arsenal,Brighton,test,1,1,D,H,False,Draw,Win,0.732399,0.168893,0.098707,0.732399
3,6a5309936cc57b7b,2024-09-15,2024,Tottenham,Arsenal,test,0,1,A,H,False,Win,Loss,0.341539,0.281824,0.376637,0.376637
4,fd02e32795fd8d5c,2024-09-22,2024,Manchester City,Arsenal,test,2,2,D,H,False,Draw,Loss,0.145856,0.187329,0.666815,0.666815
5,34092d323ebc45db,2024-09-28,2024,Arsenal,Leicester,test,4,2,H,H,True,Win,Win,0.890298,0.070449,0.039253,0.890298
6,6827a933bacd765d,2024-10-05,2024,Arsenal,Southampton,test,3,1,H,H,True,Win,Win,0.907175,0.064002,0.028823,0.907175
7,dc6e05ee150fd243,2024-10-19,2024,Bournemouth,Arsenal,test,2,0,H,A,False,Loss,Win,0.427498,0.291562,0.280940,0.427498
8,afa13334f919ec4b,2024-10-27,2024,Arsenal,Liverpool,test,2,2,D,H,False,Draw,Win,0.406932,0.285018,0.308050,0.406932
9,cbed38a7a379881d,2024-11-02,2024,Newcastle,Arsenal,test,1,0,H,A,False,Loss,Win,0.429869,0.296260,0.273871,0.429869


### league_mlp

,match_id,date,season_start,home_team,away_team,split,home_goals,away_goals,actual_result,predicted_result,correct,actual_arsenal_result,predicted_arsenal_result,p_win,p_draw,p_loss,confidence
0,519c21973743d19a,2024-08-17,2024,Arsenal,Wolves,test,2,0,H,H,True,Win,Win,0.856154,0.105788,0.038058,0.856154
1,464d111b44344fcd,2024-08-24,2024,Aston Villa,Arsenal,test,0,2,A,A,True,Win,Win,0.680467,0.173180,0.146353,0.680467
2,e2a41210638f2075,2024-08-31,2024,Arsenal,Brighton,test,1,1,D,H,False,Draw,Win,0.708722,0.184581,0.106697,0.708722
3,6a5309936cc57b7b,2024-09-15,2024,Tottenham,Arsenal,test,0,1,A,A,True,Win,Win,0.370810,0.260846,0.368344,0.370810
4,fd02e32795fd8d5c,2024-09-22,2024,Manchester City,Arsenal,test,2,2,D,H,False,Draw,Loss,0.133039,0.177979,0.688982,0.688982
5,34092d323ebc45db,2024-09-28,2024,Arsenal,Leicester,test,4,2,H,H,True,Win,Win,0.782658,0.144808,0.072534,0.782658
6,6827a933bacd765d,2024-10-05,2024,Arsenal,Southampton,test,3,1,H,H,True,Win,Win,0.805993,0.137089,0.056918,0.805993
7,dc6e05ee150fd243,2024-10-19,2024,Bournemouth,Arsenal,test,2,0,H,A,False,Loss,Win,0.587625,0.223369,0.189005,0.587625
8,afa13334f919ec4b,2024-10-27,2024,Arsenal,Liverpool,test,2,2,D,H,False,Draw,Win,0.407709,0.247420,0.344870,0.407709
9,cbed38a7a379881d,2024-11-02,2024,Newcastle,Arsenal,test,1,0,H,A,False,Loss,Win,0.521150,0.248570,0.230280,0.521150


### full_finetune_mlp

,match_id,date,season_start,home_team,away_team,split,home_goals,away_goals,actual_result,predicted_result,correct,actual_arsenal_result,predicted_arsenal_result,p_win,p_draw,p_loss,confidence
0,519c21973743d19a,2024-08-17,2024,Arsenal,Wolves,test,2,0,H,H,True,Win,Win,0.858578,0.104604,0.036818,0.858578
1,464d111b44344fcd,2024-08-24,2024,Aston Villa,Arsenal,test,0,2,A,A,True,Win,Win,0.676253,0.175165,0.148581,0.676253
2,e2a41210638f2075,2024-08-31,2024,Arsenal,Brighton,test,1,1,D,H,False,Draw,Win,0.714690,0.182376,0.102934,0.714690
3,6a5309936cc57b7b,2024-09-15,2024,Tottenham,Arsenal,test,0,1,A,H,False,Win,Loss,0.364662,0.262405,0.372933,0.372933
4,fd02e32795fd8d5c,2024-09-22,2024,Manchester City,Arsenal,test,2,2,D,H,False,Draw,Loss,0.127415,0.176746,0.695839,0.695839
5,34092d323ebc45db,2024-09-28,2024,Arsenal,Leicester,test,4,2,H,H,True,Win,Win,0.788827,0.141757,0.069416,0.788827
6,6827a933bacd765d,2024-10-05,2024,Arsenal,Southampton,test,3,1,H,H,True,Win,Win,0.811670,0.134051,0.054279,0.811670
7,dc6e05ee150fd243,2024-10-19,2024,Bournemouth,Arsenal,test,2,0,H,A,False,Loss,Win,0.584299,0.225057,0.190644,0.584299
8,afa13334f919ec4b,2024-10-27,2024,Arsenal,Liverpool,test,2,2,D,H,False,Draw,Win,0.413178,0.248525,0.338297,0.413178
9,cbed38a7a379881d,2024-11-02,2024,Newcastle,Arsenal,test,1,0,H,A,False,Loss,Win,0.518705,0.249581,0.231714,0.518705


... (4 outputs left out)

### league_random_forest

,match_id,date,season_start,home_team,away_team,split,home_goals,away_goals,actual_result,predicted_result,correct,actual_arsenal_result,predicted_arsenal_result,p_win,p_draw,p_loss,confidence
0,519c21973743d19a,2024-08-17,2024,Arsenal,Wolves,test,2,0,H,H,True,Win,Win,0.703805,0.223517,0.072677,0.703805
1,464d111b44344fcd,2024-08-24,2024,Aston Villa,Arsenal,test,0,2,A,A,True,Win,Win,0.579161,0.255077,0.165762,0.579161
2,e2a41210638f2075,2024-08-31,2024,Arsenal,Brighton,test,1,1,D,H,False,Draw,Win,0.808970,0.127336,0.063694,0.808970
3,6a5309936cc57b7b,2024-09-15,2024,Tottenham,Arsenal,test,0,1,A,H,False,Win,Loss,0.357873,0.279372,0.362755,0.362755
4,fd02e32795fd8d5c,2024-09-22,2024,Manchester City,Arsenal,test,2,2,D,H,False,Draw,Loss,0.196375,0.221112,0.582512,0.582512
5,34092d323ebc45db,2024-09-28,2024,Arsenal,Leicester,test,4,2,H,H,True,Win,Win,0.755741,0.132359,0.111901,0.755741
6,6827a933bacd765d,2024-10-05,2024,Arsenal,Southampton,test,3,1,H,H,True,Win,Win,0.772317,0.140367,0.087317,0.772317
7,dc6e05ee150fd243,2024-10-19,2024,Bournemouth,Arsenal,test,2,0,H,A,False,Loss,Win,0.623819,0.217930,0.158251,0.623819
8,afa13334f919ec4b,2024-10-27,2024,Arsenal,Liverpool,test,2,2,D,H,False,Draw,Win,0.440231,0.249467,0.310301,0.440231
9,cbed38a7a379881d,2024-11-02,2024,Newcastle,Arsenal,test,1,0,H,A,False,Loss,Win,0.554417,0.224684,0.220899,0.554417


### league_dummy

,match_id,date,season_start,home_team,away_team,split,home_goals,away_goals,actual_result,predicted_result,correct,actual_arsenal_result,predicted_arsenal_result,p_win,p_draw,p_loss,confidence
0,519c21973743d19a,2024-08-17,2024,Arsenal,Wolves,test,2,0,H,H,True,Win,Win,0.455055,0.25063,0.294315,0.455055
1,464d111b44344fcd,2024-08-24,2024,Aston Villa,Arsenal,test,0,2,A,H,False,Win,Loss,0.294315,0.25063,0.455055,0.455055
2,e2a41210638f2075,2024-08-31,2024,Arsenal,Brighton,test,1,1,D,H,False,Draw,Win,0.455055,0.25063,0.294315,0.455055
3,6a5309936cc57b7b,2024-09-15,2024,Tottenham,Arsenal,test,0,1,A,H,False,Win,Loss,0.294315,0.25063,0.455055,0.455055
4,fd02e32795fd8d5c,2024-09-22,2024,Manchester City,Arsenal,test,2,2,D,H,False,Draw,Loss,0.294315,0.25063,0.455055,0.455055
5,34092d323ebc45db,2024-09-28,2024,Arsenal,Leicester,test,4,2,H,H,True,Win,Win,0.455055,0.25063,0.294315,0.455055
6,6827a933bacd765d,2024-10-05,2024,Arsenal,Southampton,test,3,1,H,H,True,Win,Win,0.455055,0.25063,0.294315,0.455055
7,dc6e05ee150fd243,2024-10-19,2024,Bournemouth,Arsenal,test,2,0,H,H,True,Loss,Loss,0.294315,0.25063,0.455055,0.455055
8,afa13334f919ec4b,2024-10-27,2024,Arsenal,Liverpool,test,2,2,D,H,False,Draw,Win,0.455055,0.25063,0.294315,0.455055
9,cbed38a7a379881d,2024-11-02,2024,Newcastle,Arsenal,test,1,0,H,H,True,Loss,Loss,0.294315,0.25063,0.455055,0.455055


In [4]:
for rel,digest in record['artifact_sha256'].items(): assert train.digest(RUN/rel)==digest
print('All source experiment artifacts remain unchanged; all seven model verdict tables exported.')

All source experiment artifacts remain unchanged; all seven model verdict tables exported.
